# Tutorial 1: circuits, simulators and sampling (Qiskit)

*Near-term Quantum Computing 2026-2027, tutorial 1.*

A state vector $|\psi\rangle$ is a useful description of the state of a quantum system.
However, a quantum computer does not hand us the state $|\psi\rangle$ it prepares.
Each run of a quantum circuit (a *shot*) ends with a measurement that returns one bitstring.
Everything we learn about $|\psi\rangle$, such as the expectation value of an observable
$\langle \psi | O | \psi \rangle$, is estimated from a finite number of shots and comes with a
statistical error. Simulators, on the other hand, can compute the state vector exactly. The
trade-off: the memory required to store a general state of $n$ qubits scales as $2^n$. The time
required for computing the state also scales exponentially.

In this tutorial you will learn to:

1. build and draw circuits in Qiskit;
2. compute the exact state and outcome probabilities with a statevector simulator;
3. sample measurement outcomes, as a quantum device does;
4. estimate Pauli expectation values from samples, with error bars, and compare them with exact values;
5. measure in other bases, by rotating before the measurement.

**How this tutorial works.**

- **Part A** (about 45 minutes): we code together. Type the code yourself in the empty
  cells as we go.
- **Part B**: exercises, alone or in pairs. Exercises 1 and 2 are the core; 3 and 4 go
  further. Finish what you can in class, and the rest at home.
- Questions marked **(paper)** are for pen and paper. Answer them *before* writing code:
  predicting a result and then checking it in simulation is the skill we practise all course.

Documentation: [Qiskit API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit), [Qiskit Aer simulators](https://qiskit.github.io/qiskit-aer/).

## Setup

In Google Colab, run the next cell first to install the packages. If you work on your own
computer, install the same packages in your Python environment instead.

In [ ]:
%pip install -q "qiskit~=2.5.2" "qiskit-aer~=0.17.2" pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Part A: coding together

In this part, we learn the basics of Qiskit by building a circuit that prepares the Bell state

$$
|\Phi^+\rangle = \mathrm{CNOT}_{0\to 1}\, H_0\, |00\rangle = \frac{|00\rangle + |11\rangle}{\sqrt 2},
$$

measuring it in different bases, and estimating the CHSH quantity

$$
S = \langle A_0 B_0 \rangle + \langle A_0 B_1 \rangle + \langle A_1 B_0 \rangle - \langle A_1 B_1 \rangle ,
$$

where $A_0, A_1$ are two observables measured on qubit 0 and $B_0, B_1$ two observables measured
on qubit 1, all with outcomes $\pm 1$. Any local hidden-variable model gives $|S| \le 2$ (the
CHSH inequality), while quantum mechanics allows values up to $2\sqrt{2}$.

### A1. A first circuit

In Qiskit, `QuantumCircuit(n)` creates a circuit on $n$ qubits, all starting in $|0\rangle$.
Gate methods append gates in time order: `qc.h(0)` is a Hadamard on qubit 0, `qc.cx(0, 1)` a
CNOT with control 0 and target 1. All gate methods are listed in the
[`QuantumCircuit` reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit).

In [ ]:
from qiskit import QuantumCircuit

# Build the Bell-state circuit and draw it
# YOUR CODE HERE

**Native gates.** As we saw in L1, a device applies only a few *native gates*, set by how
it drives its qubits. On most platforms, the rotations $R_X(\theta)$ and $R_Y(\theta)$ are
native, while $H$ is decomposed into native gates (and rotations about $Z$ are done
virtually, without a pulse). Here we can avoid the Hadamard: $R_Y(\pi/2)\,|0\rangle =
(|0\rangle + |1\rangle)/\sqrt2$, just like $H|0\rangle$. From here on we use this version of
the Bell circuit. In Qiskit, $R_Y(\theta)$ on qubit `q` is `qc.ry(theta, q)`.

In [ ]:
# Rebuild `bell` with R_Y(pi/2) in place of H, and draw it
# YOUR CODE HERE

### A2. Exact simulation: the statevector

A statevector simulator stores all $2^n$ complex amplitudes of
$|\psi\rangle = \sum_x \psi_x |x\rangle$ (the sum runs over all bitstrings $x$) and applies
each gate as a matrix-vector multiplication.

`Statevector(circuit)` returns the final state of a circuit started in $|0\cdots0\rangle$
([reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector)).

In [ ]:
from qiskit.quantum_info import Statevector

# Compute the final state of `bell` and display it
# YOUR CODE HERE

The outcome probabilities follow from the Born rule, $p(x) = |\langle x|\psi\rangle|^2 = |\psi_x|^2$:

In [ ]:
# Print the probability of each outcome
# YOUR CODE HERE

> ⚠️ **Bit ordering.** Qiskit writes bitstrings with qubit 0 on the **right**: `'01'` means
> $b_1 = 0$ and $b_0 = 1$ ("little-endian"). Many textbooks, and these notebooks, write kets
> with qubit 0 first, $|b_0 b_1\rangle$: Qiskit's `'01'` is then $|10\rangle$. See the
> [bit-ordering guide](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering).

**Check.** Predict the output of a 2-qubit circuit with a single `x(0)` gate, then run it.

In [ ]:
# YOUR CODE HERE

**Cost.** Each amplitude is a complex number of 16 bytes (double precision), so $n$ qubits
need $16 \cdot 2^n$ bytes: 30 qubits already take 16 GiB. You will measure this cost in
Exercise 4.

### A3. Sampling: what a quantum device returns

Measuring all qubits in the computational basis returns one bitstring $x$ with probability
$p(x)$, and destroys the superposition. To learn about $|\psi\rangle$ we repeat preparation
and measurement $N$ times ($N$ *shots*) and count how often each outcome occurs, $n_x$. The
frequency $f_x = n_x / N$ estimates $p(x)$.

`measure_all()` measures every qubit and stores the results in a classical register.
[`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)
behaves like a device: it returns counts, not amplitudes. Getting them takes four steps:

```python
simulator = AerSimulator()                    # 1. the backend
job = simulator.run(circuit, shots=N)         # 2. a job
result = job.result()                         # 3. the result
counts = result.get_counts()                  # 4. the counts
```

| Step | Object | What it does |
| --- | --- | --- |
| `AerSimulator()` | backend | Plays the role of a quantum device: it executes circuits and samples measurement outcomes. On real hardware, this is the object that represents the device. |
| `.run(circuit, shots=N)` | job | Submits the circuit for $N$ shots. On hardware a job waits in a queue; the simulator runs it right away. |
| `.result()` | result | Waits until the job is done and returns everything it produced (data and metadata such as the number of shots). |
| `.get_counts()` | `dict` | Extracts the counts $n_x$: a dictionary from bitstrings to the number of shots that gave them, e.g. `{"00": 52, "11": 48}`. |

The same four steps work on any Qiskit backend, real or simulated; only the first line changes.

In [ ]:
# Make a copy of `bell` with measurements on all qubits, and draw it
# YOUR CODE HERE

In [ ]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

# Run 100 shots and print the counts
# YOUR CODE HERE

Run the last cell again: the counts change from run to run. Two observations:

- Each count $n_x$ is a binomial random variable, so the frequency $f_x$ has standard
  deviation $\sqrt{p(x)\,(1-p(x))/N}$. Halving the statistical error costs four times as
  many shots.
- Outcomes `01` and `10` never occur: the two qubits always give the same result. But a
  classical coin that sets both qubits to `00` or `11` would give the same counts:
  measurements in this basis alone cannot reveal entanglement. A5 shows what can.

### A4. Expectation values: exact versus sampled

The Pauli operator $Z$ has eigenvalue $+1$ on $|0\rangle$ and $-1$ on $|1\rangle$. So measuring
qubit $q$ with result $b_q \in \{0, 1\}$ is a measurement of $Z_q$ with outcome $(-1)^{b_q}$.
Likewise, $Z_0 Z_1$ has outcome $(-1)^{b_0 + b_1}$, the *parity* of the two bits, and

$$
\langle Z_0 Z_1 \rangle = \sum_x p(x)\,(-1)^{x_0 + x_1}.
$$

**Exact.** With the statevector we do not need to sample: $\langle\psi|P|\psi\rangle$ can be
computed directly from the amplitudes. This gives the exact expectation value, the limit of
infinitely many shots, and it is only available in simulation.

Qiskit provides this shortcut: `psi.expectation_value(Pauli("Z"), [0])` computes
$\langle\psi|Z_0|\psi\rangle$, where the second argument lists the qubits the operator acts on
([`Pauli` reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Pauli)).

In [ ]:
from qiskit.quantum_info import Pauli

# Print the exact <Z0> and <Z0 Z1> for the Bell state
# YOUR CODE HERE

**From samples.** With $N$ shots we get outcomes $s_1, \dots, s_N \in \{+1, -1\}$. We estimate
$\langle P \rangle$ by the sample mean, with statistical uncertainty (standard error)

$$
\bar s = \frac{1}{N} \sum_{i=1}^{N} s_i,
\qquad
\Delta \bar s = \sqrt{\frac{1 - \bar s^{\,2}}{N}}.
$$

The error follows in one step. Since $s^2 = 1$ for every outcome, a single shot has variance
$\mathrm{Var}(s) = \mathbb{E}[s^2] - \mathbb{E}[s]^2 = 1 - \langle P \rangle^2$, and the mean of
$N$ independent shots has variance $\mathrm{Var}(s)/N$. The exact $\langle P \rangle$ is
unknown, so we plug in $\bar s$. Let us write a function that computes both.

Remember to reverse each Qiskit bitstring, so that `bits[q]` is the outcome of qubit `q`.

In [ ]:
def estimate_z_product(counts, qubits):
    """Estimate <Z_q ...> over the qubits in `qubits` from a counts dictionary.

    Returns the sample mean and its standard error."""
    # YOUR CODE HERE
    return mean, error

In [ ]:
# Estimate <Z0>, <Z1> and <Z0 Z1> from `counts`
# YOUR CODE HERE

**Think.** The estimate of $\langle Z_0 \rangle$ scatters around the exact value 0 with an error
of about $1/\sqrt{N} = 0.1$. But the estimate of $\langle Z_0 Z_1 \rangle$ is exactly 1, with
zero error bar. Why is there no shot noise at all?

*Your answer:*

### A5. Measuring in other bases, and the CHSH inequality

A device measures only in the computational ($Z$) basis. To measure another observable, we
rotate the state first: applying a unitary $U$ and then measuring $Z$ measures $U^\dagger Z U$, since

$$
\langle\psi|U^\dagger Z U|\psi\rangle = \langle\psi'|Z|\psi'\rangle, \qquad |\psi'\rangle = U|\psi\rangle .
$$

We use rotations about the $Y$ axis. Applying $R_Y(-\phi)$ and then measuring $Z$ measures

$$
Z_\phi = R_Y(\phi)\, Z\, R_Y(-\phi) = \cos\phi\, Z + \sin\phi\, X ,
$$

the Pauli operator along the axis at angle $\phi$ from $Z$ in the $XZ$ plane of the Bloch sphere:
$\phi = 0$ measures $Z$, and $\phi = \pi/2$ measures $X$. The functions below measure qubit $q$
along $Z_{\phi_q}$ and estimate the correlator $\langle Z_{\phi_0} Z_{\phi_1} \cdots \rangle$.

In Qiskit, $R_Y(-\phi)$ on qubit `q` is `qc.ry(-phi, q)`.

In [ ]:
def measure_along(circuit, angles):
    """Copy of `circuit` that measures qubit q along cos(phi) Z + sin(phi) X, with phi = angles[q]."""
    measured = circuit.copy()
    # YOUR CODE HERE
    measured.measure_all()
    return measured


def estimate_correlator(circuit, angles, shots):
    """Estimate the correlator <Z_phi0 Z_phi1 ...>, with phi_q = angles[q], from `shots` shots.

    Returns the sample mean and its standard error."""
    # YOUR CODE HERE


measure_along(bell, [np.pi / 2, np.pi / 4]).draw("mpl")

**Predict, then check.** What are $\langle X_0 X_1 \rangle$, $\langle Z_0 X_1 \rangle$ and
$\langle X_0 Z_1 \rangle$ for $|\Phi^+\rangle$?

*Your answer:*

In [ ]:
for label, angles in [("X0 X1", [np.pi / 2, np.pi / 2]),
                      ("Z0 X1", [0, np.pi / 2]),
                      ("X0 Z1", [np.pi / 2, 0])]:
    # mean, error: estimate of the correlator from 1000 shots
    # YOUR CODE HERE
    print(f"<{label}> = {mean:+.3f} ± {error:.3f}")

**The CHSH inequality.** Alice (qubit 0) measures $A_0 = Z$ or $A_1 = X$; Bob (qubit 1)
measures $B_0 = Z_{\pi/4} = (Z + X)/\sqrt2$ or $B_1 = Z_{-\pi/4} = (Z - X)/\sqrt2$. With these settings, the CHSH inequality reads

$$
S = \langle Z \otimes Z_{\pi/4} \rangle + \langle Z \otimes Z_{-\pi/4} \rangle
  + \langle X \otimes Z_{\pi/4} \rangle - \langle X \otimes Z_{-\pi/4} \rangle \le 2
  \quad \text{(local hidden variables)}.
$$

For
$|\Phi^+\rangle$, $\langle Z_\alpha \otimes Z_\beta \rangle = \cos(\alpha - \beta)$, so three
correlators are $1/\sqrt2$, one is $-1/\sqrt2$, and $S = 2\sqrt2$: the quantum maximum.

Now the outcomes of each setting are random, and the error bars are no longer zero. The four
settings are measured in separate runs, so their estimates are independent and their
variances add: $\Delta S = \sqrt{\sum_k (\Delta \bar s_k)^2}$, summed over the four settings.

In [ ]:
CHSH_SETTINGS = [            # (Alice's angle, Bob's angle, sign in S)
    (0, np.pi / 4, +1),
    (0, -np.pi / 4, +1),
    (np.pi / 2, np.pi / 4, +1),
    (np.pi / 2, -np.pi / 4, -1),
]


def estimate_chsh(circuit, shots):
    """Estimate S for a 2-qubit circuit, with `shots` shots per setting.

    Returns the estimate and its standard error."""
    # YOUR CODE HERE


S, S_error = estimate_chsh(bell, 100)
print(f"S = {S:.3f} ± {S_error:.3f}   (exact: {2 * np.sqrt(2):.3f})")

Plot the estimate of $S$ against the number of shots per setting.

In [ ]:
shots_per_setting = [10, 30, 100, 300, 1000, 3000]

# S_values, S_errors: estimate of S and its standard error, one per entry of `shots_per_setting`
# YOUR CODE HERE

plt.errorbar(shots_per_setting, S_values, yerr=S_errors, fmt="o", capsize=3, label="estimate")
plt.axhline(2, color="gray", linestyle="--", label="local hidden variables: |S| ≤ 2")
plt.axhline(2 * np.sqrt(2), color="gray", linestyle=":", label="quantum maximum: 2√2")
plt.xscale("log")
plt.xlabel("shots per setting")
plt.ylabel("S")
plt.legend();

**Think.**

1. How many shots per setting are needed for the estimate of $S$ to exceed the classical
   bound 2 by three standard errors?
2. With few shots, some estimates may lie above $2\sqrt2$. Does that contradict quantum mechanics?

*Your answer:*

### Summary: from mathematics to code

| Mathematics | Qiskit |
| --- | --- |
| $\lvert\psi\rangle = U \lvert 0\cdots0\rangle$ | `qc = QuantumCircuit(n)`, then gates such as `qc.h(0)`, `qc.cx(0, 1)` |
| amplitudes $\psi_x$ | `Statevector(qc)` |
| $p(x) = \lvert\psi_x\rvert^2$ | `Statevector(qc).probabilities_dict()` (qubit 0 is the rightmost bit) |
| $\langle\psi\vert Z_0 Z_1\vert\psi\rangle$, exact | `Statevector(qc).expectation_value(Pauli("ZZ"), [0, 1])` |
| $N$ samples from $p(x)$ | `qc.measure_all()`, then `AerSimulator().run(qc, shots=N).result().get_counts()` |
| $\bar s \pm \Delta\bar s$ | `estimate_z_product(counts, qubits)` |
| measure $U^\dagger Z U$: apply $U$, then measure $Z$ | for $Z_\phi$: `qc.ry(-phi, q)`, then measure; `measure_along(qc, angles)` |
| $\langle Z_{\phi_0} Z_{\phi_1} \rangle$ from $N$ shots | `estimate_correlator(qc, angles, N)` |

## Part B: exercises

Exercises 1 and 2 are the core of this tutorial; 3 and 4 go further. Use the functions from
Part A, and the documentation linked above.

### Exercise 1 (core): a rotated qubit, predicted and simulated

Consider the single-qubit state

$$
|\psi(\theta)\rangle = R_Y(\theta)\,|0\rangle,
\qquad
R_Y(\theta) = e^{-i\theta Y/2} =
\begin{pmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}.
$$

**(a) (paper)** Compute $p(0)$, $p(1)$ and $\langle Z \rangle$ as functions of $\theta$.

*Your answer:*

**(b) (code)** A [`Parameter`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.Parameter) is a symbolic gate angle, filled
in later with `assign_parameters`. This lets us build a circuit once and evaluate it at many
angles. Compute the exact $\langle Z \rangle$ at the 25 angles below, and plot it together
with your prediction from (a).

In [ ]:
from qiskit.circuit import Parameter

theta = Parameter("θ")
rotation = QuantumCircuit(1)
rotation.ry(theta, 0)

angles = np.linspace(0, 2 * np.pi, 25)
fine = np.linspace(0, 2 * np.pi, 200)

# exact_z: list of exact <Z> values, one per angle in `angles`
# prediction: your formula from (a), evaluated on `fine`
# YOUR CODE HERE

plt.plot(fine, prediction, label="prediction from (a)")
plt.plot(angles, exact_z, "o", label="statevector simulation")
plt.xlabel("θ")
plt.ylabel("⟨Z⟩")
plt.legend();

**(c) (code)** Now estimate $\langle Z \rangle$ at the same angles from 200 shots each, and plot
the estimates with error bars (`plt.errorbar`) on top of your prediction.
Hint: `estimate_z_product` from A4 turns counts into an estimate and its error bar.

In [ ]:
SHOTS = 200
rotation_meas = rotation.copy()
rotation_meas.measure_all()

# means, errors: lists with the estimate of <Z> and its standard error, one per angle
# YOUR CODE HERE

plt.plot(fine, prediction, label="prediction from (a)")
plt.errorbar(angles, means, yerr=errors, fmt="o", capsize=3, label=f"{SHOTS} shots")
plt.xlabel("θ")
plt.ylabel("⟨Z⟩")
plt.legend();

**(d) (paper)** Where are the error bars largest, and where do they vanish? Why? About what
fraction of the points should miss the exact curve by more than one error bar?

*Your answer:*

### Exercise 2 (core): how fast does shot noise shrink?

**(a) (code)** Check the $1/\sqrt{N}$ scaling numerically. Take $\theta = \pi/3$, so that
$\langle Z \rangle = 1/2$. For each number of shots $N$ in the list below, repeat the $N$-shot
experiment 100 times, and compute the standard deviation of the 100 estimates (`np.std`).
Plot it against $N$ on log-log axes, together with the prediction from A4.

In [ ]:
shot_numbers = [10, 30, 100, 300, 1000, 3000, 10000]
REPEATS = 100
circuit = rotation_meas.assign_parameters([np.pi / 3])

# spread: list with the standard deviation of the REPEATS estimates, one per N
# predicted: the standard error from A4, one value per N
# YOUR CODE HERE

plt.loglog(shot_numbers, spread, "o", label=f"std of {REPEATS} estimates")
plt.loglog(shot_numbers, predicted, "-", label="prediction from A4")
plt.xlabel("shots N")
plt.ylabel("statistical error")
plt.legend();

**(b) (paper)** How many shots does it take to estimate a Pauli expectation value with a
standard error of 0.01 in the worst case? And of 0.001? At 1000 shots per second (the right
order of magnitude for current cloud devices, including reset and readout), how long does
each take?

*Your answer:*

**(c) (paper) Plausible or not?**

1. A classmate estimates $\langle Z \rangle = 0.56$ from 100 shots at $\theta = \pi/3$ and
   concludes that the simulator is broken. Do you agree?
2. A report quotes $\langle Z_0 Z_1 \rangle = 0.503 \pm 0.001$ from 1000 shots. Is this error
   bar plausible as a statistical error?

*Your answer:*

### Exercise 3 (further): the $Y$ basis, and an entanglement witness

The rotations of A5 reach only observables in the $XZ$ plane of the Bloch sphere. To measure
$Y$, rotate about the $X$ axis instead: $Y = R_X(-\pi/2)\,Z\,R_X(\pi/2)$, so apply
$R_X(\pi/2)$, then measure.

**(a) (paper)** Compute $\langle X_0 X_1 \rangle$, $\langle Y_0 Y_1 \rangle$ and
$\langle Z_0 Z_1 \rangle$ for $|\Phi^+\rangle$. Use $Y|0\rangle = i|1\rangle$ and
$Y|1\rangle = -i|0\rangle$.

*Your answer:*

**(b) (code)** Complete `measure_in_basis(circuit, basis)`. It returns a copy of `circuit`
with basis-change gates and measurements, where `basis[q]` is the Pauli measured on qubit
`q`: for example `"XZ"` measures $X$ on qubit 0 and $Z$ on qubit 1. In Qiskit, $R_X(\theta)$ on
qubit `q` is `qc.rx(theta, q)`. Then estimate the three correlators of (a) from 1000 shots each, and compare
with the exact values.

In [ ]:
def measure_in_basis(circuit, basis):
    """Copy of `circuit` that measures qubit q in the basis basis[q] ('X', 'Y' or 'Z')."""
    measured = circuit.copy()
    # YOUR CODE HERE
    measured.measure_all()
    return measured


measure_in_basis(bell, "XY").draw("mpl")

For the exact values we use [`SparsePauliOp.from_sparse_list`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp),
which names the qubits explicitly: in `("XZ", [0, 1], 1.0)`, letter `i` acts on qubit `[0, 1][i]`.
This avoids the reversed order of Qiskit's Pauli labels (where `Pauli("XZ")` puts $Z$ on qubit 0).

In [ ]:
from qiskit.quantum_info import SparsePauliOp

for basis in ["XX", "YY", "ZZ"]:
    # mean, error: estimate from 1000 shots; exact: exact expectation value
    # YOUR CODE HERE
    print(f"<{basis[0]}0 {basis[1]}1>: sampled {mean:+.3f} ± {error:.3f}, exact {exact:+.3f}")

**(c) An entanglement witness, compared with CHSH.** For any product state $|a\rangle \otimes |b\rangle$,

$$
W = \langle X_0X_1 \rangle - \langle Y_0Y_1 \rangle + \langle Z_0Z_1 \rangle \le 1 ,
$$

and therefore also for any mixture of product states (separable states), because $W$ is
linear in the state. The state $|\Phi^+\rangle$ gives $W = 3$.

1. **(paper)** Prove the bound for product states. Hint: write the single-qubit states with
   their Bloch vectors $\vec a$ and $\vec b$, so that $\langle \sigma_i \otimes \sigma_j \rangle = a_i b_j$
   and $|\vec a|, |\vec b| \le 1$.
2. **(code)** Take the weakly entangled state prepared by replacing $R_Y(\pi/2)$ in the Bell
   circuit with $R_Y(0.3)$: $\cos(0.15)|00\rangle + \sin(0.15)|11\rangle$. Estimate $W$ from 100 shots per
   measurement basis, with its error bar, and $S$ with `estimate_chsh` from A5.
3. **(paper)** The exact values are $W = 1 + 2\sin 0.3 \approx 1.59$ and
   $S = \sqrt2\,(1 + \sin 0.3) \approx 1.83$. Is the state entangled? What does each test
   conclude, and what does each assume?

*Your answer:*

In [ ]:
weak = QuantumCircuit(2)
weak.ry(0.3, 0)
weak.cx(0, 1)
SHOTS = 100

# W, W_error: estimate of W and its standard error
# YOUR CODE HERE

print(f"W = {W:.3f} ± {W_error:.3f}   (exact: {1 + 2 * np.sin(0.3):.3f})")
print(f"{(W - 1) / W_error:.1f} standard errors above the separable bound")

# S, S_error: estimate of S from 100 shots per setting, and its standard error
# YOUR CODE HERE
print(f"S = {S:.3f} ± {S_error:.3f}   (exact: {np.sqrt(2) * (1 + np.sin(0.3)):.3f})")

### Exercise 4 (further): GHZ states and the cost of statevector simulation

The $n$-qubit GHZ state is $|\mathrm{GHZ}_n\rangle = \big(|0\cdots0\rangle + |1\cdots1\rangle\big)/\sqrt{2}$.
It is prepared by a Hadamard on qubit 0 followed by a chain of CNOTs: $0 \to 1$, $1 \to 2$, and so on.

**(a) (code)** Write a function `ghz(n)` that returns this circuit. Draw it for $n = 4$ and
check its statevector.

In [ ]:
def ghz(n):
    """Circuit preparing the n-qubit GHZ state."""
    # YOUR CODE HERE


# YOUR CODE HERE

**(b) (paper)** Compute $\langle Z_0 Z_1 \cdots Z_{n-1} \rangle$ and $\langle X_0 X_1 \cdots X_{n-1} \rangle$
for $|\mathrm{GHZ}_n\rangle$.

*Your answer:*

**(c) (code)** Check (b) by sampling, for $n = 2, \dots, 6$ with 1000 shots each, using
`estimate_correlator` from A5 (angle $0$ measures $Z$, angle $\pi/2$ measures $X$).

In [ ]:
for n in range(2, 7):
    # z_mean, x_mean: estimates of <Z...Z> and <X...X> from 1000 shots each
    # YOUR CODE HERE
    print(f"n = {n}:  <Z...Z> = {z_mean:+.3f},  <X...X> = {x_mean:+.3f}")

**(d) (code)** Time the `Statevector` simulation of `ghz(n)` for $n = 10, \dots, 22$ with
`time.perf_counter()`, and plot time against $n$ on a logarithmic axis. How does the time grow
with $n$? Extrapolate: how much memory and time would $n = 40$ need?

In [ ]:
import time

qubit_numbers = list(range(10, 23))

# times: list of simulation times in seconds, one per n
# YOUR CODE HERE

plt.semilogy(qubit_numbers, times, "o-")
plt.xlabel("number of qubits n")
plt.ylabel("statevector simulation time (s)");

*Your answer:*

**(e) (code)** Now measure memory instead of time. Python's built-in
[`tracemalloc`](https://docs.python.org/3/library/tracemalloc.html) module records the memory
that Python allocates while it is running, including NumPy arrays, where `Statevector` keeps
its amplitudes:

```python
tracemalloc.start()                          # start recording allocations
Statevector(ghz(n))                          # the code we want to measure
_, peak = tracemalloc.get_traced_memory()    # (current, peak) memory in bytes
tracemalloc.stop()                           # stop recording, and forget the records
```

`get_traced_memory()` returns two numbers: the memory still in use now (`current`), and the
largest amount in use at any moment since `start()` (`peak`). We need the peak: the state
computed in the second line is not stored in a variable, so its memory is freed as soon as
the line has run. Only memory allocated after `start()` is counted, so the result is the
memory used by the simulation alone, not by the rest of the notebook.

Record the peak memory for $n = 10, \dots, 22$, and plot it on a logarithmic axis together
with the size of one state vector, $16 \cdot 2^n$ bytes (A2). How does the memory grow with
$n$? Why can the peak be larger than one state vector?

In [ ]:
import tracemalloc

# peaks: list of peak memory in bytes, one per n in `qubit_numbers`
# YOUR CODE HERE

plt.semilogy(qubit_numbers, peaks, "o-", label="peak memory (tracemalloc)")
plt.semilogy(qubit_numbers, [16 * 2**n for n in qubit_numbers], "--", label="one state vector: 16 · 2ⁿ bytes")
plt.xlabel("number of qubits n")
plt.ylabel("memory (bytes)")
plt.legend();

*Your answer:*

### Bonus exercise: native gates

Most superconducting devices, including IQM's, have two native gates: $\mathrm{PRX}(\theta, \varphi)$,
a rotation by $\theta$ about an axis in the $xy$-plane of the Bloch sphere at angle $\varphi$
from the $x$-axis (L1), and CZ. $R_X(\theta) = \mathrm{PRX}(\theta, 0)$ and
$R_Y(\theta) = \mathrm{PRX}(\theta, \pi/2)$ are special cases.

**(a) (paper)** Which of the circuits built in this tutorial are already written in these
native gates?

*Your answer:*

**(b) (code)** Rewrite the others in native gates, and check with `Statevector` that they
prepare the same states. In Qiskit, $\mathrm{PRX}(\theta, \varphi)$ on qubit `q` is
`qc.r(theta, phi, q)`, and CZ is `qc.cz(q1, q2)`.

In [ ]:
# YOUR CODE HERE

## Wrap-up

- A statevector simulator gives exact amplitudes, probabilities and expectation values, at a
  memory cost of $2^n$ amplitudes.
- A device, or a shot-based simulator, gives samples. Expectation values are estimated from
  them with a statistical error $\sqrt{(1 - \langle P \rangle^2)/N}$, which shrinks only as
  $1/\sqrt{N}$.
- To measure an observable other than $Z$, rotate first and then measure in $Z$. This is how
  we estimated $S$ for $|\Phi^+\rangle$, and saw it exceed the classical bound by several
  standard errors.

Next week we add *device noise*: a second kind of error, which shifts the expected value itself
instead of just adding statistical error around it.